# Analyse des produits — Online Retail II

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.load_data import charger_ventes_en_ligne
from src.clean_transactions import nettoyer_transactions

donnees = charger_ventes_en_ligne()
donnees = nettoyer_transactions(donnees)
donnees.shape

(1067371, 14)

## Construction de la table produits

In [2]:
donnees["gross_line_value"] = donnees["line_revenue"].clip(lower=0)
donnees["cancelled_line_value"] = (-donnees["line_revenue"]).clip(lower=0)
donnees["gross_quantity"] = donnees["quantity"].clip(lower=0)
donnees["cancelled_quantity"] = (-donnees["quantity"]).clip(lower=0)

produits = donnees.groupby(["stock_code", "description_clean"], as_index=False).agg(
    first_sale_date=("invoice_date", "min"),
    last_sale_date=("invoice_date", "max"),
    total_quantity=("quantity", "sum"),
    gross_quantity=("gross_quantity", "sum"),
    cancelled_quantity=("cancelled_quantity", "sum"),
    number_of_orders=("invoice_no", "nunique"),
    number_of_customers=("customer_id", "nunique"),
    gross_revenue=("gross_line_value", "sum"),
    cancelled_revenue=("cancelled_line_value", "sum"),
    net_revenue=("line_revenue", "sum"),
)

produits["cancellation_rate"] = produits["cancelled_quantity"] / produits[
    "gross_quantity"
].replace(0, pd.NA)

produits.head()

,stock_code,description_clean,first_sale_date,last_sale_date,total_quantity,gross_quantity,cancelled_quantity,number_of_orders,number_of_customers,gross_revenue,cancelled_revenue,net_revenue,cancellation_rate
0,10002,INFLATABLE POLITICAL GLOBE,2009-12-01 09:08:00,2011-04-28 15:05:00,7790,9016,1226,371,164,7097.90,883.55,6214.35,0.13598
1,10002R,ROBOT PENCIL SHARPNER,2009-12-02 14:43:00,2010-01-25 17:36:00,4,4,0,3,0,20.57,0.00,20.57,0.0
2,10080,CHECK,2011-11-10 10:53:00,2011-11-10 10:53:00,22,22,0,1,0,0.00,0.00,0.00,0.0
3,10080,GROOVY CACTUS INFLATABLE,2009-12-02 16:02:00,2011-11-21 17:04:00,575,575,0,29,23,129.29,0.00,129.29,0.0
4,10109,BENDY COLOUR PENCILS,2009-12-03 12:31:00,2010-02-04 13:47:00,0,4,4,2,1,1.68,0.00,1.68,1.0


## Top 10 produits par quantité nette vendue

In [3]:
produits.sort_values("total_quantity", ascending=False).head(10)[
    ["stock_code", "description_clean", "total_quantity", "net_revenue"]
]

,stock_code,description_clean,total_quantity,net_revenue
4854,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,108545,24898.22
5833,85123A,WHITE HANGING HEART T-LIGHT HOLDER,92453,253541.51
5405,84879,ASSORTED COLOUR BIRD ORNAMENT,81306,131413.85
5794,85099B,JUMBO BAG RED RETROSPOT,77671,146689.00
138,17003,BROCADE RING PURSE,70700,14743.41
1636,21977,PACK OF 60 PINK PAISLEY CAKE CASES,56575,28373.68
5596,84991,60 TEATIME FAIRY CAKE CASES,54366,27216.27
798,21212,PACK OF 72 RETROSPOT CAKE CASES,49344,28688.28
797,21212,PACK OF 72 RETRO SPOT CAKE CASES,46106,23759.26
2296,22492,MINI PAINT SET VINTAGE,44124,28102.14


## Top 10 produits par chiffre d'affaires net

In [4]:
produits.sort_values("net_revenue", ascending=False).head(10)[
    ["stock_code", "description_clean", "net_revenue", "total_quantity"]
]

,stock_code,description_clean,net_revenue,total_quantity
2200,22423,REGENCY CAKESTAND 3 TIER,327813.65,26096
6511,DOT,DOTCOM POSTAGE,322647.47,2938
5833,85123A,WHITE HANGING HEART T-LIGHT HOLDER,253541.51,92453
4285,47566,PARTY BUNTING,147948.50,28141
5794,85099B,JUMBO BAG RED RETROSPOT,146689.00,77671
5405,84879,ASSORTED COLOUR BIRD ORNAMENT,131413.85,81306
1743,22086,PAPER CHAIN KIT 50'S CHRISTMAS,121662.14,35925
6514,POST,POSTAGE,112341.00,10108
4707,79321,CHILLI LIGHTS,84854.16,16651
4921,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,73814.72,13868


## Prix unitaires les plus élevés

In [5]:
prix_par_produit = (
    donnees.groupby(["stock_code", "description_clean"])["unit_price"]
    .max()
    .sort_values(ascending=False)
)
prix_par_produit.head(10)

stock_code    description_clean                  
M             MANUAL                                 38970.00
BANK CHARGES  BANK CHARGES                           18910.69
AMAZONFEE     AMAZON FEE                             17836.46
B             ADJUST BAD DEBT                        11062.06
POST          POSTAGE                                 8142.75
ADJUST        ADJUSTMENT BY JOHN ON 26/01/2010 17     5117.03
DOT           DOTCOM POSTAGE                          4505.17
D             DISCOUNT                                1867.86
84016         FLAG OF ST GEORGE CAR FLAG              1157.15
CRUK          CRUK COMMISSION                         1100.44
Name: unit_price, dtype: float64

## Produits les plus souvent annulés

Limité aux produits ayant un volume brut significatif (≥ 50 unités), pour
éviter qu'un produit vendu 2 fois et annulé 1 fois affiche un taux de 50%
trompeur.

In [6]:
produits_significatifs = produits[produits["gross_quantity"] >= 50]
produits_significatifs.sort_values("cancellation_rate", ascending=False).head(10)[
    ["stock_code", "description_clean", "cancellation_rate", "gross_quantity"]
]

,stock_code,description_clean,cancellation_rate,gross_quantity
173,17061,ASSORTED SHAPED STENCIL FOR HENNA,23.038462,52
6487,D,DISCOUNT,15.653061,196
217,18010,ASSORTED FRAGRANCE BATH CONFETTI,12.0625,64
4265,47504G,ENGLISH ROSE TAPE MEASURE,10.760274,292
63,16050,TEATIME PENCIL WITH RUBBER,5.582781,151
4264,47504F,ENGLISH ROSE TORCH,5.522727,528
3856,35071,ASSORTED SANTA CHRISTMAS DECORATION,4.0,50
5131,84613C,BLUE NEW BAROQUE FLOCK CANDLESTICK,3.015873,63
133,16256C,HEARTS PENCIL/RUBBER+5 MINI PENCILS,2.47929,169
3849,35004S,SET OF 3 SILVER FLYING DUCKS,2.144828,145


## Classification ABC

Catégorie A : produits générant les premiers 80% du CA net. Catégorie B :
jusqu'à 95%. Catégorie C : les derniers 5%.

In [7]:
produits_tries = produits.sort_values("net_revenue", ascending=False).reset_index(
    drop=True
)
produits_tries["cumulative_share"] = (
    produits_tries["net_revenue"].cumsum() / produits_tries["net_revenue"].sum()
)


def classer_abc(part_cumulee: float) -> str:
    if part_cumulee <= 0.80:
        return "A"
    if part_cumulee <= 0.95:
        return "B"
    return "C"


produits_tries["abc_class"] = produits_tries["cumulative_share"].apply(classer_abc)
produits_tries["abc_class"].value_counts()

abc_class
C    4357
B    1112
A    1059
Name: count, dtype: int64

## Part du chiffre d'affaires par classe ABC

In [8]:
produits_tries.groupby("abc_class")["net_revenue"].sum() / produits_tries[
    "net_revenue"
].sum()

abc_class
A    0.799824
B    0.150150
C    0.050026
Name: net_revenue, dtype: float64

## Prochaines étapes (hors périmètre de cette phase)

- **Market Basket Analysis** (produits achetés ensemble) : nécessite
  `mlxtend`, prévu dans une phase dédiée.
- **Saisonnalité / analyse XYZ** (régularité de la demande) : nécessite le
  détail mensuel par produit, prévu plus tard si le temps le permet.
- **Popularité par pays** : croisement produit × pays, prévu plus tard.

## Export

In [9]:
produits_tries.to_csv(Path("../data/processed/products.csv"), index=False)
print("Fichier exporté : data/processed/products.csv")

Fichier exporté : data/processed/products.csv
